### Librerías utilizadas

| Librería | Para qué la usamos |
|---|---|
| `numpy` | Cálculo numérico y manejo de arreglos. |
| `pandas` | Cargar y manipular la tabla de datos (el `DataFrame`). |
| `matplotlib` | Generar los gráficos (*boxplots*). |
| `sklearn.preprocessing` | Las herramientas de codificación (`LabelEncoder`, `OneHotEncoder`) y escalamiento (`StandardScaler`). |

`np.random.seed(42)` fija la semilla aleatoria: garantiza que cualquier proceso con azar
dé **siempre el mismo resultado**, lo que asegura la *reproducibilidad* exigida en la fase.

In [ ]:
import numpy as np                  # cálculo numérico y operaciones vectorizadas
import pandas as pd                 # estructuras tabulares: Series y DataFrame
import matplotlib.pyplot as plt     # gráficos de control
from sklearn import preprocessing    # LabelEncoder y OneHotEncoder
from sklearn.preprocessing import StandardScaler   # estandarización z-score

# Reproducibilidad: fijamos la semilla aleatoria del entorno
np.random.seed(42)

%matplotlib inline

## Configuración del entorno y de las rutas

Antes de cargar nada conviene comprobar el entorno y dejar declaradas las rutas.
Se usan rutas **relativas** a la raíz del repositorio, coherentes con la
estructura creada en la Fase 1: una ruta absoluta como `C:/Users/...` funciona en
un solo computador del mundo.

In [ ]:
from pathlib import Path      # manejo de rutas independiente del sistema operativo
import sys
cwd = Path.cwd()
# sys.version trae la versión completa con fecha de compilación;
# split()[0] deja solo el número, que es lo único que hay que comprobar.
print("Python:", sys.version.split()[0])
for lib, mod in [("numpy", np), ("pandas", pd)]:
    print(f"{lib:8}:", mod.__version__)

cwd = Path.cwd()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists() or (p / "requirements.txt").exists()), cwd)
print(f"\nRaíz detectada del proyecto: {ROOT.resolve()}")
# Estructura del proyecto. parents=True crea las carpetas intermedias;
# exist_ok=True evita el error si ya existen.
DIR_CRUDO = ROOT / "Data/Raw"            # datos originales: nunca se modifican
DIR_PROCESADO = ROOT / "Data/Processed"  # resultado del pipeline
DIR_DOCS = ROOT / "docs"                 # diccionario, bitácora y metadatos
for carpeta in (DIR_CRUDO, DIR_PROCESADO, DIR_DOCS):
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVO = DIR_CRUDO / "listings.csv.gz"
print("Archivo esperado en:", ARCHIVO.resolve())

### Procedencia del conjunto de datos

| Campo | Valor |
| --- | --- |
| Título | Datos Airbnb Santiago Junio 2026 |
| Autor | Inside Airbnb |
| Plataforma | Inside Airbnb |
| Enlace | `https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/data/reviews.csv.gz` |
| Estructura esperada | 18300 filas x 90 columnas |
| Unidad de observación | Precio por noche |


In [ ]:
Path.cwd().parent.parent

In [ ]:
# Reemplaza el nombre por el de tu archivo real
# Buscar el dataset desde la carpeta superior del proyecto
base = Path.cwd().parent.parent

archivo = next(
    base.rglob("listings.csv.gz"),
    None
)

if archivo is None:
    raise FileNotFoundError(
        f"No se encontró el dataset dentro de:\n{base}"
    )

print("Dataset encontrado en:")
print(archivo)

df = pd.read_csv(archivo)

print(f"\nDataset cargado correctamente")
print(f"Dimensiones: {df.shape}")

df.head()

## 1. Obtención de los datos

**Qué hace este paso.** Lee el archivo CSV y lo carga en un `DataFrame` (la tabla con la
que trabajaremos).

**Por qué con una función.** Encapsulamos la lectura en `cargar_datos(ruta)` con un bloque
`try / except`. Así, si el archivo no existe, en lugar de un error técnico confuso, el
usuario recibe un mensaje claro indicando qué revisar. La función, además, imprime las
dimensiones cargadas, lo que sirve como primera verificación de que el archivo se leyó bien.

In [ ]:
from pathlib import Path
import pandas as pd

# El notebook está en F2 y el dataset está en F1/data/raw
base = Path.cwd().parent.parent
ARCHIVO = next(
    base.rglob("listings.csv.gz"),
    None
)
print("Directorio actual:")
print(Path.cwd())

print("\nRuta del archivo:")
print(ARCHIVO.resolve())

print("\n¿Existe el archivo?")
print(ARCHIVO.exists())

# Validación
if not ARCHIVO.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo en:\n{ARCHIVO.resolve()}"
    )

# Cargar dataset
df_crudo = pd.read_csv(
    ARCHIVO,
    na_values=["N/A"]
)

# Mantener copia original
df = df_crudo.copy()

print("\nDatos cargados correctamente")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

df.head()

Mostramos las primeras filas para confirmar visualmente que las columnas se cargaron correctamente:

`head()` muestra las primeras cinco filas. Es la comprobación más barata que existe: confirma que el separador se interpretó bien, que los nombres de columna son los esperados y que los valores no quedaron corridos de columna.



## 2. Exploración inicial

**Qué hace este paso.** Antes de modificar nada, miramos el estado original de los datos.
Esto nos da la "línea base" contra la cual verificaremos después cada transformación.

La función `explorar_dataframe` reporta cuatro cosas:
1. **Dimensiones** (`shape`): cuántas filas y columnas hay.
2. **Tipos de datos** (`dtypes`): qué columnas son numéricas y cuáles son texto. Las de
   texto (`object`) son las que tendremos que codificar más adelante.
3. **Valores nulos** (`isnull().sum()`): cuántos datos faltan en cada columna. Aquí
   detectaremos que `bmi` tiene huecos.
4. **Estadísticos descriptivos** (`describe()`): media, mínimo, máximo, etc., de las
   variables numéricas, útil para detectar rangos raros o valores atípicos.

In [ ]:
def explorar_dataframe(df):
    """Resumen exploratorio: dimensiones, tipos, nulos y estadisticos descriptivos."""
    print("Dimensiones:", df.shape)
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isnull().sum())
    print("\nEstadisticos descriptivos (variables numericas):")
    return df.describe()


explorar_dataframe(df)

**Decisión sobre variables relevantes**

Para determinar las variables que serán objeto de estudio, se extrae los nombres de todas las variables, para determinar cuales serán objeto de estudio y cuales serán descartadas, quedando un respaldo en la data/raw, pero no pasando a las siguientes fases

In [ ]:
print(df.columns)

**Decisión Final**

Tras una discusión en grupo, se determinó que las siguientes variables serán las que se estudiarán en este proyecto: (`"hosts_time_as_host_years","neighbourhood_cleansed","property_type","room_type","accommodates", "bathrooms", "bedrooms", "beds", 'price_quote_price_per_night','number_of_reviews', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value'`)

**"hosts_time_as_host_years"**: número de años que el anfitrión ha utilizado la plataforma de AirBnb.

**"neighbourhood_cleansed"**: Comuna del hospedaje.

**"property_type","room_type","accommodates", "bathrooms", "bedrooms", "beds"**: Características fundamentales del hospedaje, tipo de propiedad, tipo de habitación, número de huespedes, baños, habitaciones y camas.

**'price_quote_price_per_night'**: Precio del hospedaje por noche, ya transformada a a un float64, equivalente a la columna (`'price'`)

**'number_of_reviews', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value'**: variables de reseñas en múltiples aspectos del departamento de usuarios que se han hospedado en el departamento/casa (etc.).

In [ ]:
#estas serán las columnas objeto de estudio
columnas_de_interes = ["hosts_time_as_host_years","neighbourhood_cleansed","property_type","room_type","accommodates", "bathrooms",
                      "bedrooms", "beds", 'price_quote_price_per_night','number_of_reviews', 'review_scores_rating', 'review_scores_accuracy', 
                       'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value' ]

In [ ]:
df = df[columnas_de_interes]

In [ ]:
explorar_dataframe(df)

### El diccionario de variables

Antes de limpiar hay que declarar **qué es cada variable**. Esta es la decisión
que determina todo el preprocesamiento posterior.


| Rol analítico | Qué preprocesamiento exige |
| --- | --- |
| **Continua** | Detección de atípicos, imputación, escalamiento |
| **Binaria** | Codificación 0/1 y revisión de desbalance |
| **Nominal** | *One-hot encoding* |
| **Ordinal** | Codificación con el orden declarado explícitamente |
| **Identificador** | Verificar unicidad y excluir del análisis |
| **Objetivo** | Se preserva sin transformar en esta fase |

In [ ]:
df.head()

In [ ]:
# El diccionario se DECLARA con lo que se sabe del dominio y se COMPLETA con lo
# observado en el archivo: no se afirma nada que no se verifique.
DECLARADO = [
    ("hosts_time_as_host_years",                "ordinal", "numero de años como anfitrion"),
    ("neighbourhood_cleansed",            "nominal",       "Comuna"),
    ("property_type",               "nominal",      "Tipo de propiedad"),
    ("room_type",      "nominal",       "Habitación o casa/depa entero"),
    ("accommodates",     "ordinal",       "Nro de huespedes"),
    ("bathrooms",      "ordinal",       "Nro de baños"),
    ("bedrooms",         "ordinal",       "Nro de habitaciones"),
    ("beds",    "ordinal",       "Nro de camas"),
    ("price_quote_price_per_night",               "objetivo",      "Precio por noche"),
    ("number_of_reviews",    "ordinal",       "Nro de reseñas"),
    ("review_scores_rating",            "continua",      "1 a 5 reseña"),
    ("review_scores_accuracy",            "continua",      "1 a 5 reseña"),
    ("review_scores_cleanliness",            "continua",      "1 a 5 reseña"),
    ("review_scores_checkin",            "continua",      "1 a 5 reseña"),
    ("review_scores_communication",            "continua",      "1 a 5 reseña"),
    ("review_scores_location",            "continua",      "1 a 5 reseña"),
    ("review_scores_value",            "continua",      "1 a 5 reseña"),
]
diccionario = pd.DataFrame(DECLARADO, columns=["variable", "rol", "descripcion"])

# Columnas OBSERVADAS: se leen del archivo, no se escriben a mano
diccionario["dtype"] = [str(df_crudo[v].dtype) for v in diccionario["variable"]]
# nunique() cuenta valores distintos: distingue una binaria de una continua
diccionario["n_unicos"] = [int(df_crudo[v].nunique()) for v in diccionario["variable"]]
# Sobre booleanos, mean() devuelve la PROPORCIÓN de True: por 100 da el porcentaje
diccionario["pct_nulos"] = [round(df_crudo[v].isna().mean() * 100, 1)
                            for v in diccionario["variable"]]

# El diccionario es un entregable, no una tabla de paso: se guarda
diccionario.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False)
diccionario

### Valores atípicos: medir antes de decidir

La sección siguiente imputa con la mediana «porque hay valores extremos». Esa
afirmación hay que **demostrarla**, no enunciarla. Se usa el criterio del rango
intercuartílico:

$$\text{atípico si} \quad x < Q_1 - 1{,}5 \cdot \text{RIC} \quad \text{o} \quad x > Q_3 + 1{,}5 \cdot \text{RIC}$$

In [ ]:
def detectar_atipicos_iqr(serie, factor=1.5):
    """Identifica valores atípicos por el criterio del rango intercuartílico.

    Retorna
    -------
    tuple(np.ndarray, float, float)
        Máscara booleana de atípicos, límite inferior y límite superior.
    """
    # dropna() es obligatorio: un NaN propagaría y devolvería NaN como cuartil
    valores = serie.dropna().to_numpy(dtype="float64")

    # np.percentile con una lista devuelve varios percentiles de una vez
    q1, q3 = np.percentile(valores, [25, 75])
    ric = q3 - q1                                  # rango intercuartílico
    limite_inf, limite_sup = q1 - factor * ric, q3 + factor * ric

    # El operador | es el "o" elemento a elemento de pandas. Con 'or' fallaría:
    # 'or' espera un único booleano, no una serie completa.
    mascara = (serie < limite_inf) | (serie > limite_sup)
    # Comparar con NaN da False pero deja NA: fillna(False) lo hace explícito
    return mascara.fillna(False).to_numpy(), limite_inf, limite_sup


filas = []
for col in ["hosts_time_as_host_years", "accommodates", "bathrooms", "bedrooms", "beds", "number_of_reviews", "price_quote_price_per_night", "review_scores_value", "review_scores_location", "review_scores_communication", "review_scores_checkin", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_rating"]:
    mascara, inf, sup = detectar_atipicos_iqr(df[col])
    filas.append({
        "variable": col,
        "limite_inf": round(inf, 2),
        "limite_sup": round(sup, 2),
        "n_atipicos": int(mascara.sum()),          # sobre booleanos, sum() cuenta True
        "pct_atipicos": round(mascara.mean() * 100, 2),
        "media": round(df[col].mean(), 2),
        "mediana": round(df[col].median(), 2),
    })
pd.DataFrame(filas)

## 3. Limpieza de datos

Limpiar significa dejar los datos completos y consistentes. Tomamos dos decisiones, cada
una justificada técnicamente:


**Imputar "hosts_time_as_host_years", "accommodates", "bathrooms", "bedrooms", "beds", "number_of_reviews", "review_scores_value", "review_scores_location", "review_scores_communication", "review_scores_checkin", "review_scores_accuracy", "review_scores_cleanliness" y "review_scores_rating" con la mediana.** "Imputar" es rellenar huecos con un valor representativo.
Tenemos dos opciones:
- La **media** (promedio): se ve arrastrada por los valores extremos.
- La **mediana** (valor central): es **robusta** ante valores atípicos.
 La función `imputar_nulos_numericos` permite
elegir la estrategia con un parámetro e informa cuántos nulos rellenó y con qué valor
(trazabilidad).

**Eliminar filas N/A de "price_quote_price_per_night"**: Como se quiere mantener la mayor certeza de los resultados, las filas que presenten valores N/A serán eliminadas del dataset procesado.

In [ ]:
def imputar_nulos_numericos(df, columna, estrategia="mediana"):
    """
    Imputa los valores nulos de una columna numerica.

    Parametros
    ----------
    df : pd.DataFrame
    columna : str
        Columna numerica a imputar.
    estrategia : str
        'media' o 'mediana'.

    Retorna
    -------
    pd.DataFrame
        DataFrame con la columna imputada.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")
    # isnull() da True/False por celda; sum() cuenta los True.
    # Se calcula ANTES de imputar: después ya no habría nulos que contar.
    n_nulos = int(df[columna].isnull().sum())
    if estrategia == "media":
        valor = df[columna].mean()
    elif estrategia == "mediana":
        valor = df[columna].median()
    else:
        raise ValueError("estrategia debe ser 'media' o 'mediana'")
    df = df.copy()
    df[columna] = df[columna].fillna(valor)
    print(f"'{columna}': {n_nulos} nulos imputados con la {estrategia} = {valor:.2f}")
    return df

**Antes** de limpiar, confirmamos dónde están los nulos:

In [ ]:
df.isnull().sum()

Imputamos los nulos de `["bathrooms","bedrooms","beds","review_scores_rating","review_scores_accuracy","review_scores_cleanliness","review_scores_checkin","review_scores_communication","review_scores_location","review_scores_value"]` con la mediana:

In [ ]:
is_null = ["bathrooms","bedrooms","beds","review_scores_rating","review_scores_accuracy","review_scores_cleanliness","review_scores_checkin","review_scores_communication","review_scores_location","review_scores_value"]

In [ ]:
for x in is_null:
    df = imputar_nulos_numericos(df, x, estrategia="mediana")
df

Para mantener la variable objetivo intacta, los nulos de la columna de `price_quote_price_per_night` se tratarán eliminando las filas en los cuales se encuentren estos.

In [ ]:
df.dropna(subset=['price_quote_price_per_night'], inplace = True)
df

In [ ]:
df.isnull().sum()

In [ ]:
ARCHIVO_PROCESADO = DIR_PROCESADO / "airbnb_procesado_limpieza.csv"

# index=False evita agregar una columna con el número de fila, que no es un dato
df.to_csv(ARCHIVO_PROCESADO, index=False)

# Verificación de ida y vuelta: se relee lo guardado y se comprueba que coincide.
# Si el archivo se escribió mal, es preferible descubrirlo ahora.
releido = pd.read_csv(ARCHIVO_PROCESADO)
assert releido.shape == df.shape, "El archivo releído no coincide en dimensiones"
print(f"Guardado y verificado: {ARCHIVO_PROCESADO}")
print(f"Tamaño en disco: {ARCHIVO_PROCESADO.stat().st_size / 1024:.1f} kB")

## 4. Transformación: codificación One-Hot

Los modelos solo entienden números, no texto. Por eso hay que convertir las variables de
texto en números. Pero **cómo** las convertimos importa:

- `neighbourhood_cleansed`, `property_type` y `room_type` son

### El patrón `fit` / `transform`

Casi todas las herramientas de scikit-learn funcionan en dos tiempos:
- **`fit`** = *aprender*: la herramienta mira los datos y guarda lo que necesita (por
  ejemplo, qué categorías existen).
- **`transform`** = *aplicar*: usa lo aprendido para convertir los datos.

Dentro de la función `codificar_one_hot` este patrón aparece **dos veces** (una con
`LabelEncoder` y otra con `OneHotEncoder`). Paso a paso, con el ejemplo `["Yes","No","Yes","No"]`:

1. **`le = preprocessing.LabelEncoder()`** — crea la herramienta que convierte texto en
   enteros. Aún está vacía.
2. **`le.fit(datos)`** — aprende las categorías y les asigna un número en orden alfabético:
   `"No" → 0`, `"Yes" → 1`.
3. **`le.transform(datos)`** — aplica lo aprendido → `[1, 0, 1, 0]`. Esto ya es *Label
   Encoding*.
4. **`ohe = preprocessing.OneHotEncoder()`** — crea la segunda herramienta, para One-Hot.
5. **`d = datos_codificados.reshape(-1, 1)`** — reorganiza los datos de una fila a una
   columna, porque el `OneHotEncoder` exige formato de tabla (2 dimensiones). El `-1`
   significa "calcula tú las filas" y el `1` significa "una columna".
6. **`ohe.fit(d)` y `ohe.transform(d).toarray()`** — expande cada entero en columnas
   binarias. `.toarray()` convierte el resultado a un arreglo normal.
7. **`pd.DataFrame(...)` + `pd.concat(...)`** — empaqueta las nuevas columnas con nombres
   legibles y las une al `DataFrame`, eliminando la columna original.

**Mejoras respecto al código original:** la función (a) asigna los nombres legibles
directamente, en lugar de renombrar por número de posición; (b) conserva el índice original
(`index=df.index`) para que la unión no desalinee filas; y (c) **valida** que la cantidad
de nombres coincida con la cantidad de categorías, avisando con un error claro si no.

In [ ]:
def codificar_one_hot(df, columna, nombres_columnas):
    """
    Codifica una variable categorica nominal aplicando LabelEncoder y luego
    OneHotEncoder (patron fit/transform de scikit-learn).

    Parametros
    ----------
    df : pd.DataFrame
    columna : str
        Columna categorica a codificar.
    nombres_columnas : list[str]
        Nombres de las nuevas columnas binarias. El orden debe coincidir con el
        orden alfabetico de las categorias (criterio interno de LabelEncoder).

    Retorna
    -------
    pd.DataFrame
        DataFrame con las nuevas columnas one-hot y sin la columna original.

    Lanza
    -----
    KeyError
        Si la columna no existe.
    ValueError
        Si el numero de nombres no coincide con el numero de categorias.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")

    # Paso 1-3: LabelEncoder aprende las categorias y las convierte a enteros
    le = preprocessing.LabelEncoder()
    datos = df[columna]
    le.fit(datos)
    datos_codificados = le.transform(datos)

    # Verificacion: los nombres deben corresponder a las categorias detectadas
    if len(nombres_columnas) != len(le.classes_):
        raise ValueError(
            f"Se esperaban {len(le.classes_)} nombres para '{columna}' "
            f"({list(le.classes_)}), pero se recibieron {len(nombres_columnas)}."
        )

    # Paso 4-6: OneHotEncoder expande los enteros a columnas binarias
    ohe = preprocessing.OneHotEncoder()
    d = datos_codificados.reshape(-1, 1)   # formato 2D requerido por el encoder
    ohe.fit(d)
    matriz = ohe.transform(d).toarray()

    # Paso 7: empaquetamos con nombres legibles y conservamos el indice original
    nuevas = pd.DataFrame(matriz, columns=nombres_columnas, index=df.index).astype(int)

    df = df.drop(columns=[columna]).reset_index(drop=True)
    nuevas = nuevas.reset_index(drop=True)
    return pd.concat([df, nuevas], axis=1)

### a. `neighbourhood_cleansed`
Categorías comunas

In [ ]:
# Los nombres van en el ORDEN ALFABÉTICO de las categorías, que es el criterio
# interno de LabelEncoder. Si se invierten, las columnas
# quedan mal rotuladas y ningún error lo advierte.
lista_comunas = df["neighbourhood_cleansed"].unique().tolist()
lista_comunas.sort()
index = 0
for x in lista_comunas:
    lista_comunas[index] = "comuna_" + x
    index +=1
print(lista_comunas)

In [ ]:
df = codificar_one_hot(df, "neighbourhood_cleansed", lista_comunas)
df.head()

### b. `property_type`

Categorías tipo de propiedad

In [ ]:
lista_property_type = df["property_type"].unique().tolist()
lista_property_type.sort()
index = 0
for x in lista_property_type:
    loop_aux = x.replace(" ", "_")
    lista_property_type[index] = "tipo_" + loop_aux
    index +=1
print(lista_property_type)

In [ ]:
df = codificar_one_hot(df, "property_type", lista_property_type)
df.head()

### c. `room_type`


Categorías tipo de habitación

In [ ]:
lista_room_type = df["room_type"].unique().tolist()
lista_room_type.sort()
index = 0
for x in lista_room_type:
    loop_aux = x.replace(" ", "_")
    lista_room_type[index] = "habitacion_" + loop_aux
    index +=1
print(lista_room_type)

In [ ]:
df = codificar_one_hot(df, "room_type", lista_room_type)
df.head()

In [ ]:
ARCHIVO_PROCESADO = DIR_PROCESADO / "airbnb_procesado_limpieza_y_transformación.csv"

# index=False evita agregar una columna con el número de fila, que no es un dato
df.to_csv(ARCHIVO_PROCESADO, index=False)

# Verificación de ida y vuelta: se relee lo guardado y se comprueba que coincide.
# Si el archivo se escribió mal, es preferible descubrirlo ahora.
releido = pd.read_csv(ARCHIVO_PROCESADO)
assert releido.shape == df.shape, "El archivo releído no coincide en dimensiones"
print(f"Guardado y verificado: {ARCHIVO_PROCESADO}")
print(f"Tamaño en disco: {ARCHIVO_PROCESADO.stat().st_size / 1024:.1f} kB")

## 5. Escalamiento (estandarización)

**Qué es.** Estandarizar (*z-score*) transforma una variable para que tenga **media 0 y
desviación estándar 1**, mediante la fórmula:

$$ z = \frac{x - \mu}{\sigma} $$

donde $\mu$ es la media y $\sigma$ la desviación estándar de la columna.

**Por qué.** las Variables están en escalas muy
distintas. Algunos modelos (KNN, SVM, regresión regularizada) se basan en distancias y darían
más peso a la variable de números más grandes solo por su escala. Estandarizar las pone en
condiciones comparables.

**Qué NO escalamos.** Solo estandarizamos las variables **continuas** (`hosts_time_as_host_years`,
`accommodates`, `bathrooms`,'beds', entre otras). No tocamos las columnas binarias (las one-hot) ni la variable objetivo `price_quote_price_per_night`, porque estandarizar valores 0/1 o el
*target* distorsiona su interpretación. Esta es una corrección importante frente a aplicar
el escalador a toda la tabla.

La función devuelve también el `scaler` ajustado, por si más adelante hay que aplicar la
misma transformación a datos nuevos.

In [ ]:
from sklearn.preprocessing import RobustScaler
def escalar_caracteristicas(df, columnas):
    """
    Estandariza (z-score) las columnas indicadas con StandardScaler.

    Retorna el DataFrame transformado y el objeto scaler ajustado (para poder
    reutilizarlo sobre datos nuevos sin recalcular las estadisticas).
    """
    # copy() evita modificar el DataFrame recibido: quien llama decide si
    # reemplaza el original o conserva ambos.
    df = df.copy()
    scaler = RobustScaler()
    # fit_transform hace dos cosas: aprende media y desviación (fit) y aplica
    # la transformación (transform). Se devuelve el scaler para poder aplicar
    # EXACTAMENTE la misma transformación a datos nuevos con .transform().
    df[columnas] = scaler.fit_transform(df[columnas])
    return df, scaler

Tras escalar, la media debe quedar ≈ 0 y la desviación estándar ≈ 1:

In [ ]:
# Solo se escalan las continuas: las binarias y las codificadas ya están en una
# escala comparable, y escalarlas destruiría su interpretación como 0/1.
columnas_continuas = ["hosts_time_as_host_years", "accommodates", "bathrooms", "bedrooms", "beds", "number_of_reviews", "review_scores_value", "review_scores_location", "review_scores_communication", "review_scores_checkin", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_rating"]
df, scaler = escalar_caracteristicas(df, columnas_continuas)
df[columnas_continuas].describe()

## 6. Validación técnica y verificación del código

Verificar significa demostrar, con evidencia, que el resultado es correcto. La función
`validar_dataset` ejecuta varias comprobaciones y usa `assert`: si una condición no se
cumple, el notebook se detiene con un error, dejando constancia de la falla.

Comprobamos:
1. **Integridad** — que no queden valores nulos.
2. **Consistencia** — que todas las columnas sean numéricas (no quedó texto sin codificar).
3. **Coherencia** — que cada grupo one-hot sume exactamente 1 por fila: cada paciente
   pertenece a **una sola** categoría dentro de cada variable.
4. **Duplicados** — cuántas filas repetidas hay.
5. **Dimensiones** — la forma final de la tabla.

In [ ]:
def validar_dataset(df, grupos_one_hot=None):
    """
    Comprueba integridad, consistencia y coherencia del dataset final.
    Lanza AssertionError si alguna comprobacion falla.
    """
    print("VALIDACION DEL DATASET FINAL")
    print("-" * 45)

    # 1. Integridad: sin valores nulos
    nulos = int(df.isnull().sum().sum())
    assert nulos == 0, f"Quedan {nulos} valores nulos."
    print(f"[OK] Sin valores nulos (total = {nulos})")

    # 2. Consistencia: todas las columnas son numericas
    no_numericas = df.select_dtypes(exclude=[np.number]).columns.tolist()
    assert not no_numericas, f"Columnas no numericas: {no_numericas}"
    print("[OK] Todas las columnas son numericas")

    # 3. Coherencia: cada grupo one-hot suma 1 por fila (una sola categoria activa)
    if grupos_one_hot:
        for nombre, cols in grupos_one_hot.items():
            suma = df[cols].sum(axis=1)
            assert (suma == 1).all(), f"El grupo '{nombre}' no suma 1 en todas las filas."
            print(f"[OK] Grupo one-hot '{nombre}' coherente")

    # 4. Duplicados
    dup = int(df.duplicated().sum())
    print(f"[INFO] Filas duplicadas: {dup}")

    # 5. Dimensiones finales
    print(f"[INFO] Dimensiones finales: {df.shape}")
    return True




## Conclusiones y trazabilidad

El *pipeline* dejó el *dataset* sin valores nulos, con todas las variables en formato
numérico y las continuas estandarizadas. Cada paso se implementó como una función
documentada y reutilizable (`cargar_datos`, `explorar_dataframe`, `imputar_nulos_numericos`,
`codificar_one_hot`, `escalar_caracteristicas`, `validar_dataset`), garantizando un flujo
modular, reproducible y verificable.

**Trazabilidad con el repositorio (F2):** este notebook se ubica en la carpeta `F2/` del
repositorio GitHub; el `README` documenta las dependencias (`numpy`, `pandas`,
`scikit-learn`, `matplotlib`) y las instrucciones de ejecución. El historial de *commits*
refleja el avance del *pipeline* descrito en el informe técnico de la Fase 2.

---

## 7. Recursividad: aplanar los metadatos del proyecto

Todo el procesamiento anterior fue **estructurado**: secuencias, condicionales y
bucles. Esta sección introduce la **recursividad**, que es la herramienta natural
cuando la estructura tiene profundidad desconocida. El indicador ID 2.1 la
contempla explícitamente.

El caso es real y no decorativo: los metadatos del proyecto forman un diccionario
anidado y, para exportarlos como tabla, hay que convertirlos en pares
clave-valor. Un bucle no sirve porque no se sabe cuántos niveles hay.

**Anatomía de la función recursiva:**
- *Caso base*: el valor no es un diccionario → se devuelve el par clave-valor.
- *Caso recursivo*: el valor es un diccionario → la función se llama a sí misma
  un nivel más abajo, arrastrando el prefijo de la ruta.

In [ ]:
# Los metadatos del proyecto, con la estructura anidada que corresponde
METADATOS = {
    "proyecto": {
        "nombre": "datos airbnb",
        "fase": "F2",
        "semilla": 42,
    },
    "datos": {
        "archivo": "listings.csv.gz",
        "fuente": {
            "plataforma": "InsideAirbnb",
            "autor": "InsideAirbnb",
            "url": "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/data/reviews.csv.gz",
        },
    },
    "decisiones": {
        "imputacion": {"datos_continuos": "mediana"},
        "escalador": "Robust",
        "categoricas": ["neighbourhood_cleansed", "property_type", "room type"],
    },
}


def aplanar(estructura, prefijo="", separador="."):
    """Aplana un diccionario anidado en uno de un solo nivel.

    Ejemplo
    -------
    >>> aplanar({"a": {"b": 1, "c": {"d": 2}}})
    {'a.b': 1, 'a.c.d': 2}
    """
    plano = {}
    for clave, valor in estructura.items():
        # Se construye la ruta completa: en el primer nivel no hay prefijo
        ruta = f"{prefijo}{separador}{clave}" if prefijo else str(clave)

        if isinstance(valor, dict):
            # CASO RECURSIVO: la función se llama a sí misma un nivel más abajo
            plano.update(aplanar(valor, ruta, separador))
        elif isinstance(valor, (list, tuple)):
            # Las secuencias se convierten en texto legible para la tabla
            plano[ruta] = ", ".join(str(v) for v in valor)
        else:
            # CASO BASE: el valor es simple, se devuelve el par
            plano[ruta] = valor
    return plano


metadatos_planos = aplanar(METADATOS)
print(f"El diccionario anidado se aplanó en {len(metadatos_planos)} pares.\n")
for clave, valor in metadatos_planos.items():
    print(f"  {clave:<28} {str(valor)[:52]}")

In [ ]:
# Verificación de la función recursiva: casos normal, profundo y límite
assert aplanar({"a": 1}) == {"a": 1}, "Caso plano"
assert aplanar({"a": {"b": 1}}) == {"a.b": 1}, "Un nivel de anidamiento"
assert aplanar({"a": {"b": {"c": {"d": 4}}}}) == {"a.b.c.d": 4}, "Anidamiento profundo"
assert aplanar({}) == {}, "Caso límite: diccionario vacío"
assert aplanar({"x": [1, 2, 3]}) == {"x": "1, 2, 3"}, "Listas convertidas a texto"
print("[OK] La función recursiva pasa las cinco pruebas.")

---

## 8. Persistencia y trazabilidad

El resultado debe quedar guardado y documentado. Esta sección produce los
artefactos que se versionan en el repositorio y se citan en el informe.

In [ ]:
ARCHIVO_PROCESADO = DIR_PROCESADO / "airbnb_procesado.csv"

# index=False evita agregar una columna con el número de fila, que no es un dato
df.to_csv(ARCHIVO_PROCESADO, index=False)

# Verificación de ida y vuelta: se relee lo guardado y se comprueba que coincide.
# Si el archivo se escribió mal, es preferible descubrirlo ahora.
releido = pd.read_csv(ARCHIVO_PROCESADO)
assert releido.shape == df.shape, "El archivo releído no coincide en dimensiones"
print(f"Guardado y verificado: {ARCHIVO_PROCESADO}")
print(f"Tamaño en disco: {ARCHIVO_PROCESADO.stat().st_size / 1024:.1f} kB")

# Los metadatos aplanados con la función recursiva de la sección 7
pd.DataFrame(list(metadatos_planos.items()), columns=["clave", "valor"]).to_csv(
    DIR_DOCS / "metadatos_proyecto.csv", index=False)
print(f"Metadatos exportados a {DIR_DOCS}/metadatos_proyecto.csv")

In [ ]:
# Resumen comparativo entre el punto de partida y el resultado.
# Esta tabla responde de una vez a la pregunta "qué cambió".
resumen = pd.DataFrame([
    {"indicador": "Filas", "inicial": df_crudo.shape[0], "final": df.shape[0]},
    {"indicador": "Columnas", "inicial": df_crudo.shape[1], "final": df.shape[1]},
    {"indicador": "Valores nulos",
     "inicial": int(df_crudo.isna().sum().sum()), "final": int(df.isna().sum().sum())},
    {"indicador": "Columnas de texto",
     "inicial": int((df_crudo.dtypes == "object").sum()),
     "final": int((df.dtypes == "object").sum())},
    {"indicador": "Filas duplicadas",
     "inicial": int(df_crudo.duplicated().sum()), "final": int(df.duplicated().sum())},
])
resumen